In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np

from dataset.dataset import (
    anonymize_dataset,
    create_dataset,
    generate_increasing_datasets,
    join_dataset,
    split_dataset,
)
from dataset.utils import link_dir
from plotting import plot_metrics
from training import *


# Parameters

In [ ]:
n = 3

# Goal
Benchmark the model "yolo11-cls" model on an increasing quantity of data.

The dataset has been already tested and the model too (KFold cross validation).

Just change the training dataset quantity but keep the same test dataset
- 20% -> 40% -> 60% -> 80% -> 100

### Create datasets

In [ ]:
quantities = [0.2, 0.4, 0.6, 0.8, 1.0]
path = "dataset/benchmark"
path_csv = "dataset/benchmark"

anonymize_dataset("dataset/source/csv/sources_kfold_train.csv")

for q in quantities:
    for i in range(n):
        create_dataset("dataset/source/csv/sources_kfold_train.csv", 
                       f"dataset/benchmark/kfold_{q}/{i}/train", 
                       f"dataset/source/csv/benchmark/kfold_{q}/{i}.csv", quantity=q, random=True)
        link_dir("dataset/main/test",
                 f"dataset/benchmark/kfold_{q}/{i}/test")


### Training

In [ ]:
def benchmark_dataset_yolo(config, datasets: list[str], names, n=3):
    # mean of (score, accuracy, recall, f1)
    histories = []
    for ds_path in datasets:
        
        results = []
        if n > 1:
            for i in range(n):
                name = os.path.basename(ds_path)

                ds = Path(ds_path, str(i))

                model = train(name, ds, config)
                _, metric = test(model, ds)

                results.append(metric[0])

            results = np.asarray(results)
            histories.append([(results.mean(), results.std())])
        else:
            name = os.path.basename(ds_path)
            model = train(name, ds_path, config)
            _, metric = test(model, ds_path)
            histories.append([(metric[0], 0)])
        
    print(histories)
    plot_metrics(histories, names, metrics=[("accuracy", "number of images")])

In [ ]:
config = {"model": "models/yolo11x-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }
# "dataset/benchmark/kfold_0.4/", "dataset/benchmark/kfold_0.6/", "dataset/benchmark/kfold_0.8/", "dataset/main/"
# "40", "60", "80", "100"

# benchmark_dataset_yolo(config, ["dataset/benchmark/kfold_0.2/"], ["20"])
benchmark_dataset_yolo(config, ["dataset/benchmark/kfold_0.2/", 
                                "dataset/benchmark/kfold_0.4/", 
                                "dataset/benchmark/kfold_0.6/", 
                                "dataset/benchmark/kfold_0.8/", 
                                "dataset/benchmark/kfold_1.0/"], 
                       ["20", "40", "60", "80", "100"])

## Benchmark Learning Curve: LogScale

In [ ]:
def logscale(start, end, num=10):
    values = np.geomspace(start, end, num=num)
    return np.unique(np.array(np.round(values) * 2, dtype=int))

In [ ]:
train_set = "dataset/source/csv/sources_kfold_train.csv"
L = logscale(1, 40, 10)
L = np.asarray(L)

#### Dataset Generation

In [ ]:
# split
[df_train, df_val] = split_dataset(train_set, [40, 10], random=True)

output_dir = "dataset/benchmark/learning_curve"
output_csv = "dataset/source/csv/benchmark/learning_curve"

create_dataset(df_val, f"{output_dir}/val", f"{output_csv}/val.csv")

for size in L:
    create_dataset(df_train, f"{output_dir}/{size}/train", f"{output_csv}/{size}.csv", 
                   size, is_random=True, is_subject_level=True)
    
    link_dir(f"{output_dir}/val",
            f"{output_dir}/{size}/val")

### Training

In [ ]:
config = {"model": "models/yolo11x-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }

dataset_path = "dataset/benchmark/learning_curve"

datasets_path = [ f"{dataset_path}/{size}" for size in L]
points = [ f"{size}" for size in L]

benchmark_dataset_yolo(config, datasets_path, points, n=1)

## Learning Curve: Cross Validation

The goal is to produce a learning curve based on cross validation.

Each fold have a validation set. For each fold we train N models (where N is the number of point)
with increasing datasets (4 -> 10 -> ...), we keep add data to the training (4 -> 4 old data + 6 new data -> 10 old + 5 new data -> ...).

After all we compute the fold average result of each point of the curve

In [ ]:
source = "dataset/source/csv/sources_kfold_train.csv"
output_dir = "dataset/benchmark/learning_curve_cv"
output_csv = "dataset/source/csv/benchmark/learning_curve_cv"
sizes = np.asarray(logscale(1, 40, 10), dtype=int)

config = {"model": "models/yolo11x-cls.pt", 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }

In [ ]:
# generate 5 fold
validations = split_dataset(source, [10, 10, 10, 10, 10], random=True)
folds = [] # [(val, train)]

for i, df_val in enumerate(validations):
    df_train_lst = [chunck for j, chunck in enumerate(validations) if j != i]
    df_train = join_dataset(df_train_lst)
    folds.append((df_val, df_train))

folds_with_increasing_dataset = []
# foreach fold generate X dataset [0 -> 80]
for df_val, df_train in folds:
    all_train_df = generate_increasing_datasets(df_train, sizes, random=True)
    folds_with_increasing_dataset.append((df_val, all_train_df))

metrics_all_points: list[list] = [[] for _ in range(len(sizes))]

for df_val, lst_df_train in folds_with_increasing_dataset:
    
    create_dataset(df_val, f"{output_dir}/val", f"{output_csv}/val.csv")
    
    for pt, df_train in enumerate(lst_df_train):
        
        # dataset creation
        name = str(df_train.shape[0])
        ds = f"{output_dir}/{name}"
        create_dataset(df_train, output_dir, f"{output_csv}/{name}.csv")
        
        # training
        model = train(name, ds, config)
        _, metric = test(model, ds)

        metrics_all_points[pt].append(metric[0])
    
    # suppresion folder learning_curve_cv
    shutil.rmtree(output_dir)

histories_mean = [np.asarray(l, dtype=int).mean() for l in metrics_all_points]
names = [f"{i}" for i in sizes]
plot_metrics(histories_mean, names, metrics=[("accuracy", "number of images")])